In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "04-inference-engine/vllm-internals/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# Block hashes and eviction order, rebuilt from vLLM

**Tier:** T0 (Python standard library only; laptop or Colab CPU).

**The one-minute version.** vLLM names every full KV block by a hash chained over the whole prefix, keeps freed
blocks cached in a doubly linked free list, and allocates from that list's head: uncached blocks first, then cached
blocks in LRU order, with each prefix chain's tail evicted before its root. This notebook rebuilds those pieces in
about 100 lines, using the names in `vllm/v1/core/kv_cache_utils.py` and `vllm/v1/core/block_pool.py` (vLLM `main`
at `5840d95`, 2026-09-25), and checks five behaviours from sections 4.3–4.6 of
[`../vllm-internals-primer.md`](../vllm-internals-primer.md).

| Here | vLLM |
|---|---|
| `H`, `NONE_HASH` | `vllm/utils/hashing.py: sha256` (pickle, then SHA-256); `kv_cache_utils.init_none_hash` (fixed seed) |
| `hash_block_tokens`, `block_hashes` | `kv_cache_utils.hash_block_tokens`, `get_request_block_hasher`, `generate_block_hash_extra_keys` |
| `Block`, `FreeQueue` | `KVCacheBlock`, `FreeKVCacheBlockQueue` |
| `BlockPool` | `BlockPool.get_new_blocks`, `touch`, `free_blocks`, `get_usage`; `BlockHashToBlockMap` |
| `longest_hit`, `admit` | `KVCacheManager.get_computed_blocks` + `allocate_slots` (one full-attention group, no scheduler) |

Simplifications: one KV cache group, no sliding window, no multimodal keys, and a block size of 4 instead of 16 so
the printouts stay short.

In [ ]:
import hashlib, pickle
from dataclasses import dataclass

BLOCK_SIZE = 4                                   # vLLM's default is 16

def H(obj) -> bytes:                             # vllm/utils/hashing.py: sha256
    return hashlib.sha256(pickle.dumps(obj, protocol=pickle.HIGHEST_PROTOCOL)).digest()

NONE_HASH = H("vllm-none-hash")                  # init_none_hash, fixed default seed

def hash_block_tokens(parent, tokens, extra_keys=None) -> bytes:
    return H((parent or NONE_HASH, tuple(tokens), extra_keys))

def block_hashes(tokens, lora_name=None, cache_salt=None) -> list:
    """Full blocks only. LoRA name on every block; cache salt on the first block only."""
    out, parent = [], None
    for start in range(0, len(tokens) - BLOCK_SIZE + 1, BLOCK_SIZE):
        keys = ([lora_name] if lora_name else []) + ([cache_salt] if start == 0 and cache_salt else [])
        parent = hash_block_tokens(parent, tokens[start:start + BLOCK_SIZE], tuple(keys) or None)
        out.append(parent)
    return out

@dataclass(eq=False)
class Block:                                     # KVCacheBlock
    block_id: int
    ref_cnt: int = 0
    block_hash: bytes = None
    prev: "Block" = None
    next: "Block" = None

class FreeQueue:                                 # FreeKVCacheBlockQueue: fake head and tail, O(1) remove
    def __init__(self, blocks):
        self.head, self.tail, self.num_free = Block(-1), Block(-1), 0
        self.head.next, self.tail.prev = self.tail, self.head
        self.append_n(blocks)
    def _insert_after(self, where, b):
        b.prev, b.next = where, where.next
        where.next.prev = where.next = b
        self.num_free += 1
    def append_n(self, blocks):                  # tail: reused last
        for b in blocks: self._insert_after(self.tail.prev, b)
    def prepend_n(self, blocks):                 # head: reused first, order kept
        for b in reversed(blocks): self._insert_after(self.head, b)
    def remove(self, b):
        b.prev.next, b.next.prev = b.next, b.prev
        b.prev = b.next = None
        self.num_free -= 1
    def popleft(self):
        assert self.head.next is not self.tail, "no free blocks"
        b = self.head.next; self.remove(b); return b
    def ids(self):
        out, b = [], self.head.next
        while b is not self.tail: out.append(b.block_id); b = b.next
        return out

class BlockPool:
    def __init__(self, num_blocks):
        self.blocks = [Block(i) for i in range(num_blocks)]
        self.free = FreeQueue(self.blocks)
        self.cached = {}                         # BlockHashToBlockMap: hash -> {block_id: block}
        self.null_block = self.free.popleft()    # block 0 is the reserved null block
    def lookup(self, h):
        d = self.cached.get(h)
        return next(iter(d.values())) if d else None
    def cache(self, b, h):
        b.block_hash = h
        self.cached.setdefault(h, {})[b.block_id] = b
    def get_new_blocks(self, n):                 # pop from the head; evict a hash the block still carries
        out = []
        for _ in range(n):
            b = self.free.popleft()
            if b.block_hash is not None:         # _maybe_evict_cached_block
                d = self.cached[b.block_hash]; del d[b.block_id]
                if not d: del self.cached[b.block_hash]
                b.block_hash = None
            b.ref_cnt = 1; out.append(b)
        return out
    def touch(self, blocks):                     # a prefix hit takes a block back from the free list
        for b in blocks:
            if b.ref_cnt == 0: self.free.remove(b)
            b.ref_cnt += 1
    def free_blocks(self, ordered):              # callers pass reversed(request_blocks)
        first, last = [], []
        for b in ordered:
            b.ref_cnt -= 1
            if b.ref_cnt == 0: (last if b.block_hash is not None else first).append(b)
        self.free.prepend_n(first)               # uncached: "LIFO reuse ... for better GPU locality"
        self.free.append_n(last)                 # cached: "FIFO reuse ... for LRU eviction behavior"
    def get_usage(self):                         # what vllm:kv_cache_usage_perc reports
        return 1 - self.free.num_free / (len(self.blocks) - 1)

def longest_hit(pool, tokens, **extra):          # get_computed_blocks: hit capped at num_tokens - 1
    hit = []
    for h in block_hashes(tokens, **extra)[: (len(tokens) - 1) // BLOCK_SIZE]:
        b = pool.lookup(h)
        if b is None: break                      # a miss implies a miss for every later block
        hit.append(b)
    return hit

def admit(pool, tokens, **extra):                # get_computed_blocks + allocate_slots for a new request
    hit = longest_hit(pool, tokens, **extra)
    pool.touch(hit)
    blocks = hit + pool.get_new_blocks(-(-len(tokens) // BLOCK_SIZE) - len(hit))
    for b, h in zip(blocks, block_hashes(tokens, **extra)):
        if b.block_hash is None: pool.cache(b, h)   # cache_full_blocks: hashed at allocation
    return blocks, len(hit) * BLOCK_SIZE

print("ready: block size", BLOCK_SIZE)

## 1. A hash names the whole prefix

Each hash is computed from its parent hash, so editing one token changes that block's hash and every hash after it,
and leaves the blocks before it untouched. Only full blocks are hashed.

*Predict before running:* a 16-token prompt has four full blocks. If token 5 (inside block 1) changes, which of the
four hashes change?

In [ ]:
prompt = list(range(100, 116))            # 16 tokens = 4 full blocks
edited = prompt.copy(); edited[5] = 999  # a token inside block 1
same = [a == b for a, b in zip(block_hashes(prompt), block_hashes(edited))]
print("block hash unchanged?", same)
assert same == [True, False, False, False]
assert len(block_hashes(prompt[:15])) == 3   # the partial fourth block has no hash yet
print("check passed: the edit in block 1 changes blocks 1-3; a partial block is not hashed")

## 2. Extra keys isolate adapters and tenants

The LoRA adapter name is an extra key on every block; `cache_salt` is added to the first block only, yet it still
changes every hash, because every later hash chains from the first. Two tenants with the same prompt but different
salts, or the same prompt under two adapters, never share a block.

In [ ]:
plain = block_hashes(prompt)
lora = block_hashes(prompt, lora_name="adapter-a")
salted = block_hashes(prompt, cache_salt="tenant-42")
assert all(a != b for a, b in zip(plain, lora))
assert all(a != b for a, b in zip(plain, salted))
print("check passed: a LoRA name or a first-block salt changes every block hash")

## 3. A fully cached prompt still computes one block

The last prompt token must run through the model to produce logits, so the hit is capped at `num_tokens - 1`, and
hits are whole blocks. *Predict:* the same 16-token prompt arrives twice. How many tokens hit the cache the second
time, and how many are recomputed? Then a 21-token prompt that starts with those 16 tokens arrives.

In [ ]:
pool = BlockPool(num_blocks=16)
_, hit_first = admit(pool, prompt)
_, hit_second = admit(pool, prompt)
_, hit_longer = admit(pool, prompt + [7, 7, 7, 7, 7])
print("hits:", hit_first, hit_second, hit_longer)
assert (hit_first, hit_second, hit_longer) == (0, 12, 16)   # (16 - 1) // 4 = 3 blocks for the repeat
print("check passed: the identical prompt recomputes one block; the longer prompt reuses all 16 tokens")

## 4. Eviction order: uncached first, then LRU, a chain's tail before its root

`KVCacheManager.free` hands the request's blocks to `BlockPool.free_blocks` in **reverse** order; blocks without a
hash go to the head of the free queue, blocks with a hash to the tail. Allocation pops from the head.

*Predict:* request X (14 tokens: three full blocks and one partial) gets blocks 1–4 from a fresh pool of 14 blocks,
then finishes. What does the free queue look like, head to tail? A second, unrelated request Y then needs 11 blocks.
Which of X's cached blocks survive?

In [ ]:
pool = BlockPool(num_blocks=14)            # blocks 1..13 usable; 0 is the null block
x_prompt = list(range(200, 214))           # 14 tokens
x_blocks, _ = admit(pool, x_prompt)
print("X holds", [b.block_id for b in x_blocks])
pool.free_blocks(reversed(x_blocks))
print("free queue after X finishes:", pool.free.ids())
assert pool.free.ids() == [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 3, 2, 1]

y_blocks, _ = admit(pool, list(range(500, 544)))   # 44 unrelated tokens = 11 blocks
print("Y got", [b.block_id for b in y_blocks], "| still free:", pool.free.ids())
survivors = [pool.lookup(h) is not None for h in block_hashes(x_prompt)]
print("X's blocks still cached (root first):", survivors)
assert survivors == [True, True, False]
assert len(longest_hit(pool, x_prompt)) * BLOCK_SIZE == 8
print("check passed: X's partial block went first, its chain tail next, its root survives for the next hit")

## 5. What `vllm:kv_cache_usage_perc` does not show

`BlockPool.get_usage` is `1 - free / (num_blocks - 1)`, and cached blocks in the free queue count as free. A replica
can report 0% usage while holding a warm prefix cache.

In [ ]:
pool = BlockPool(num_blocks=9)             # 8 usable blocks
blocks, _ = admit(pool, prompt)            # 16 tokens = 4 blocks
usage_running = pool.get_usage()
pool.free_blocks(reversed(blocks))
usage_after, still_cached = pool.get_usage(), len(pool.cached)
hit_after = len(longest_hit(pool, prompt)) * BLOCK_SIZE
print(f"usage while running {usage_running:.2f}, after finishing {usage_after:.2f}, "
      f"cached blocks {still_cached}, next hit {hit_after} tokens")
assert (usage_running, usage_after, still_cached, hit_after) == (0.5, 0.0, 4, 12)
print("check passed: usage fell to 0 while all four blocks stayed cached and reusable")

## In a design review

**Explain it in two minutes.** "vLLM hashes each full 16-token block together with its parent's hash, so a hash
names the whole prefix; LoRA names and a tenant salt are folded in, which isolates adapters and tenants. When a
request finishes, its blocks go back to a doubly linked free list in reverse order: the partial block to the head,
because nothing can ever hit it, and the hashed blocks to the tail, so they are the last to be reused. Allocation pops
the head, so cached blocks are evicted in LRU order and a chain loses its tail before its root, which is the part the
next request most likely shares. A later hit pulls a block out of the middle of the list in constant time."

**Drills.**

1. *Why evict a chain's tail before its root?* The root (system prompt, tool schemas) is shared by the most future
   requests; the tail is specific to one conversation. Freeing in reverse and appending puts the root last in line.
2. *A router reads `kv_cache_usage_perc = 0` on a replica. Is its cache cold?* Not necessarily: usage counts only
   blocks referenced by live requests (check 5). Cache affinity needs block hashes (KV events), not this gauge.
3. *Why does the partial last block go to the head?* It has no hash, so no request can hit it; reusing it first keeps
   useful cached blocks alive longer (and, per the source comment, helps GPU locality).